# Baseline 5: DINO ViT-B/16 + Demographic MLP

Self-supervised ViT backbone (DINO) counterpart to MURD-ViT. Replaces the supervised ViT-B/16 with a self-supervised ViT-B/16 (DINO) backbone. Same temporal Siamese fusion and demographic MLP head.
All other training settings (split, hyperparameters, loss, metrics) are identical to `baseline_model.ipynb`.

In [ ]:
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
import torchvision.transforms as T
import timm
import pandas as pd
import geopandas as gpd
import numpy as np
import cv2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.cluster import KMeans
import folium
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from datetime import datetime
import logging
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image

In [ ]:
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f"Seed: {SEED}")

In [ ]:
!nvidia-smi

In [ ]:
gc.collect()
torch.cuda.empty_cache()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

## Dataset

In [ ]:
class UrbanRetrofittingDataset(Dataset):
    def __init__(self, excel_file, image_dir, transform=None):
        self.data = pd.read_excel(excel_file, dtype={"Latitude": str, "Longitude": str})
        # maintains precision. Do not change this two step conversion.
        self.data = self.data.astype({"Latitude": np.longdouble, "Longitude": np.longdouble})
        self.image_dir = image_dir
        self.transform = transform
        self.label_encoder = LabelEncoder()
        self.data['Change'] = self.label_encoder.fit_transform(self.data['Change'])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        census_block = str(row['ID'])
        lat, lon, direction = row['Latitude'], row['Longitude'], row['Heading']
        start_date, end_date = row['Start'], row['End']
        pop_density_change = row['PopDen_Diff']
        change_percentage = row['PopDen_Diff_Percentage']
        label = row['Change']

        # Load image pair. Do not use f{} formatting to add longdouble number.
        start_image_path = os.path.join(self.image_dir, census_block, str(lat)+"_"+str(lon)+"_"+str(direction), f"{pd.to_datetime(start_date).strftime('%b %Y')}.png")
        end_image_path   = os.path.join(self.image_dir, census_block, str(lat)+"_"+str(lon)+"_"+str(direction), f"{pd.to_datetime(end_date).strftime('%b %Y')}.png")
        start_image = Image.open(start_image_path).convert('RGB')
        end_image   = Image.open(end_image_path).convert('RGB')

        if self.transform:
            start_image = self.transform(start_image)
            end_image   = self.transform(end_image)

        socio_economic_data = torch.tensor([pop_density_change, change_percentage], dtype=torch.float32)
        return start_image, end_image, socio_economic_data, label

## Model: DINO ViT-B/16 + Demographic MLP

In [ ]:
class DINOWithSocioEconomic(nn.Module):
    def __init__(self, num_numeric_features, num_classes):
        super(DINOWithSocioEconomic, self).__init__()
        # Shared DINO ViT-B/16 backbone (self-supervised pretrained, no classification head)
        self.backbone = timm.create_model('vit_base_patch16_224.dino', pretrained=True, num_classes=0, img_size=224)
        img_embedding_dim = self.backbone.num_features  # 768

        # Demographic embedding — mirrors MURD-ViT numeric_embed
        self.numeric_embed = nn.Sequential(
            nn.Linear(num_numeric_features, 32),
            nn.ReLU(),
            nn.Linear(32, 32)
        )

        # Classification head — mirrors MURD-ViT classifier (3×768+32 = 2336 → 256 → 5)
        self.classifier = nn.Sequential(
            nn.Linear(3 * img_embedding_dim + 32, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, start_image, end_image, numeric_feats):
        start_feat = self.backbone(start_image)
        end_feat   = self.backbone(end_image)
        diff_feat  = end_feat - start_feat
        numeric_feat = self.numeric_embed(numeric_feats)
        combined = torch.cat([start_feat, end_feat, diff_feat, numeric_feat], dim=1)
        return self.classifier(combined)

## Training and Validation Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, num_classes):
    model.train()
    total_samples = 0
    running_loss = 0.0
    correct_total = 0
    class_correct = [0] * num_classes
    class_total   = [0] * num_classes
    class_losses  = [0.0] * num_classes

    for start_imgs, end_imgs, socio_econ, labels in loader:
        start_imgs, end_imgs = start_imgs.to(device), end_imgs.to(device)
        socio_econ, labels   = socio_econ.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(start_imgs, end_imgs, socio_econ)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        running_loss  += loss.item() * batch_size
        total_samples += batch_size
        preds = outputs.argmax(dim=1)
        correct_total += (preds == labels).sum().item()

        for cls in range(num_classes):
            cls_mask = labels == cls
            if cls_mask.sum() > 0:
                cls_outputs = outputs[cls_mask]
                cls_labels  = labels[cls_mask]
                class_losses[cls]   += criterion(cls_outputs, cls_labels).item() * cls_mask.sum().item()
                class_correct[cls]  += (cls_outputs.argmax(dim=1) == cls_labels).sum().item()
                class_total[cls]    += cls_mask.sum().item()

    epoch_loss   = running_loss / total_samples
    overall_acc  = 100 * correct_total / total_samples
    class_acc    = [100 * class_correct[i] / class_total[i] if class_total[i] else 0 for i in range(num_classes)]
    class_avg_loss = [class_losses[i] / class_total[i] if class_total[i] else 0 for i in range(num_classes)]
    return epoch_loss, overall_acc, class_acc, class_avg_loss

In [ ]:
def validate_epoch(model, loader, criterion, device, num_classes):
    model.eval()
    total_samples = 0
    running_loss  = 0.0
    correct_total = 0
    class_correct = [0] * num_classes
    class_total   = [0] * num_classes
    class_losses  = [0.0] * num_classes
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for start_imgs, end_imgs, socio_econ, labels in loader:
            start_imgs, end_imgs = start_imgs.to(device), end_imgs.to(device)
            socio_econ, labels   = socio_econ.to(device), labels.to(device)

            outputs = model(start_imgs, end_imgs, socio_econ)
            loss = criterion(outputs, labels)

            batch_size = labels.size(0)
            running_loss  += loss.item() * batch_size
            total_samples += batch_size

            _, preds_top2 = outputs.topk(2, dim=1)
            correct_top2  = preds_top2.eq(labels.unsqueeze(1)).any(dim=1)
            stored_pred_batch = torch.where(correct_top2, labels, preds_top2[:, 0])

            all_preds.extend(stored_pred_batch.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            correct_total += correct_top2.sum().item()

            for cls in range(num_classes):
                cls_mask = labels == cls
                if cls_mask.sum() > 0:
                    cls_outputs = outputs[cls_mask]
                    cls_labels  = labels[cls_mask]
                    class_losses[cls]  += criterion(cls_outputs, cls_labels).item() * cls_mask.sum().item()
                    class_correct[cls] += correct_top2[cls_mask].sum().item()
                    class_total[cls]   += cls_mask.sum().item()

    epoch_loss   = running_loss / total_samples
    overall_acc  = 100 * correct_total / total_samples
    class_acc    = [100 * class_correct[i] / class_total[i] if class_total[i] else 0 for i in range(num_classes)]
    class_avg_loss = [class_losses[i] / class_total[i] if class_total[i] else 0 for i in range(num_classes)]
    return epoch_loss, overall_acc, class_acc, class_avg_loss, all_preds, all_labels

## Configuration

In [ ]:
checkpoint_dir = "../DINO_checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
MODEL_NAME      = "dino"
CHECKPOINT_PATH = f"{checkpoint_dir}/{MODEL_NAME}_epoch_30.pth"

logs_dir = "../logs"
os.makedirs(logs_dir, exist_ok=True)
LOG_FILE = f"{logs_dir}/{MODEL_NAME}.txt"

# Hyperparameters — identical to MURD-ViT
BATCH_SIZE        = 256      # change to 256 on H100
NUM_EPOCHS        = 30
LEARNING_RATE     = 0.0001
WEIGHT_DECAY_RATE = 0.00001
NUM_CLASSES       = 5

excel_File = "../labels.xlsx"
image_dir  = "../Streetview_data"

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(LOG_FILE, mode='a'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

## Data Loading and Split

In [ ]:
# DINO uses the same ImageNet normalization as MURD-ViT — no divergence
train_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def spatial_stratified_split(dataset, val_ratio=0.2, n_clusters=10):
    df = dataset.data.copy()
    df = df.dropna(subset=['Latitude', 'Longitude'])
    coords = df[['Latitude', 'Longitude']].values
    kmeans = KMeans(n_clusters=n_clusters, random_state=42)
    df['Cluster'] = kmeans.fit_predict(coords)
    train_indices, val_indices = [], []
    for cluster in df['Cluster'].unique():
        cluster_df = df[df['Cluster'] == cluster]
        for label in cluster_df['Change'].unique():
            label_df = cluster_df[cluster_df['Change'] == label]
            n_val = max(1, int(len(label_df) * val_ratio))
            val_sample   = label_df.sample(n=n_val, random_state=42)
            train_sample = label_df.drop(val_sample.index)
            train_indices.extend(train_sample.index)
            val_indices.extend(val_sample.index)
    df.reset_index(drop=True, inplace=True)
    dataset.data = df
    gdf = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']), crs='EPSG:4326')
    return train_indices, val_indices, gdf

logger.info("Data Loading Initiated!")
train_dataset = UrbanRetrofittingDataset(excel_File, image_dir, train_transform)
val_dataset   = UrbanRetrofittingDataset(excel_File, image_dir, val_transform)

train_indices, val_indices, clusters_gdf = spatial_stratified_split(dataset=train_dataset, val_ratio=0.2, n_clusters=20)
_, _, _ = spatial_stratified_split(dataset=val_dataset, val_ratio=0.2, n_clusters=20)

train_dataset = Subset(train_dataset, train_indices)
val_dataset   = Subset(val_dataset,   val_indices)
logger.info(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}")

In [ ]:
all_labels = train_dataset.dataset.data['Change']

def log_distribution(all_labels, dataset, name):
    label_counts  = Counter([all_labels[i] for i in dataset.indices])
    label_encoder = dataset.dataset.label_encoder
    logger.info(f"{name} distribution:")
    for label_idx in sorted(label_counts):
        logger.info(f"\t{label_encoder.classes_[label_idx]}: {label_counts[label_idx]}")

log_distribution(all_labels, train_dataset, "Train")
log_distribution(all_labels, val_dataset,   "Validation")

## Class Weights and Samplers

In [ ]:
train_labels = [all_labels[i] for i in train_dataset.indices]
num_classes  = len(set(train_labels))

# Manually calibrated weights — identical to MURD-ViT
class_weights_sampler = np.array([0.0036, 0.0034, 0.0150, 0.0050, 0.0060])
sample_weights = np.array([class_weights_sampler[label] for label in train_labels])

sampler = WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)

# Loss weights — manually calibrated, identical to MURD-ViT
class_weights = np.array([1.66, 7.22, 5.56, 9.22, 9.34])
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
logger.info("Class weights used for loss function:")
for label, weight in zip(train_dataset.dataset.label_encoder.classes_, class_weights):
    logger.info(f"\tClass {label}: Weight={weight:.4f}")

In [ ]:
import multiprocessing
num_workers = multiprocessing.cpu_count() // 2

logger.info(f"Batch Size: {BATCH_SIZE}, Workers: {num_workers}, pin_memory: True, LR: {LEARNING_RATE}, Wt. Decay: {WEIGHT_DECAY_RATE}")

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=num_workers, pin_memory=True)
logger.info("Training Loader is now ready!")
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True)
logger.info("Validation Loader is now ready!")

## Training Loop

In [ ]:
model     = DINOWithSocioEconomic(2, NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss(weight=weights_tensor, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY_RATE)

if os.path.exists(CHECKPOINT_PATH):
    checkpoint = torch.load(CHECKPOINT_PATH, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch          = checkpoint['epoch']
    train_acc_per_class  = checkpoint['train_acc_per_class']
    val_acc_per_class    = checkpoint['val_acc_per_class']
    train_loss_per_class = checkpoint['train_loss_per_class']
    val_loss_per_class   = checkpoint['val_loss_per_class']
    overall_train_acc    = checkpoint['overall_train_acc']
    overall_val_acc      = checkpoint['overall_val_acc']
    overall_train_loss   = checkpoint['overall_train_loss']
    overall_val_loss     = checkpoint['overall_val_loss']
    val_preds            = checkpoint['val_preds']
    val_labels           = checkpoint['val_labels']
    logger.info(f"Loaded checkpoint from epoch {start_epoch}")
else:
    start_epoch          = 0
    train_acc_per_class  = []
    val_acc_per_class    = []
    train_loss_per_class = []
    val_loss_per_class   = []
    overall_train_acc    = []
    overall_val_acc      = []
    overall_train_loss   = []
    overall_val_loss     = []
    val_preds            = None
    val_labels           = None
    logger.info("No checkpoint found — starting new training")

for epoch in range(start_epoch, NUM_EPOCHS):
    train_loss, train_overall_acc, train_class_acc, train_class_loss = train_epoch(
        model, train_loader, criterion, optimizer, device, NUM_CLASSES)
    val_loss, val_overall_acc, val_class_acc, val_class_loss, val_preds, val_labels = validate_epoch(
        model, val_loader, criterion, device, NUM_CLASSES)

    train_acc_per_class.append(train_class_acc)
    val_acc_per_class.append(val_class_acc)
    train_loss_per_class.append(train_class_loss)
    val_loss_per_class.append(val_class_loss)
    overall_train_acc.append(train_overall_acc)
    overall_val_acc.append(val_overall_acc)
    overall_train_loss.append(train_loss)
    overall_val_loss.append(val_loss)

    logger.info(f"Epoch {epoch+1}/{NUM_EPOCHS}")
    logger.info(f"Train Accuracy: {train_overall_acc:.2f}%")
    logger.info(f"Train Loss: {train_loss:.4f}")
    for idx, cls in enumerate(train_dataset.dataset.label_encoder.classes_):
        logger.info(f"\tTrain Class '{cls}': Acc={train_class_acc[idx]:.2f}%, Loss={train_class_loss[idx]:.4f}")
    logger.info(f"Validation Accuracy: {val_overall_acc:.2f}%")
    logger.info(f"Validation Loss: {val_loss:.4f}")
    for idx, cls in enumerate(train_dataset.dataset.label_encoder.classes_):
        logger.info(f"\tVal Class '{cls}': Acc={val_class_acc[idx]:.2f}%, Loss={val_class_loss[idx]:.4f}")

    if (epoch + 1) >= 18:
        CHECKPOINT_PATH = os.path.join(checkpoint_dir, f"{MODEL_NAME}_epoch_{epoch+1}.pth")
        torch.save({
            'epoch':                epoch + 1,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_acc_per_class':  train_acc_per_class,
            'val_acc_per_class':    val_acc_per_class,
            'train_loss_per_class': train_loss_per_class,
            'val_loss_per_class':   val_loss_per_class,
            'overall_train_acc':    overall_train_acc,
            'overall_val_acc':      overall_val_acc,
            'overall_train_loss':   overall_train_loss,
            'overall_val_loss':     overall_val_loss,
            'val_preds':            val_preds,
            'val_labels':           val_labels,
        }, CHECKPOINT_PATH)
        logger.info(f"Saved checkpoint to {CHECKPOINT_PATH}")

    logger.info("-" * 100)

## Evaluation

In [ ]:
classes = val_dataset.dataset.label_encoder.classes_
print(classification_report(val_labels, val_preds, target_names=classes, digits=4))

conf_matrix = confusion_matrix(val_labels, val_preds, normalize='true')
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt=".2f", cmap="Blues", xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title(f"Confusion Matrix — {MODEL_NAME}")
plt.show()

In [ ]:
classes_list = ['No Change', 'Re-building', 'Re-capital', 'Re-inhabitation', 'Re-transportation']
color_map = {
    'No Change':         "#c2bf04",
    'Re-building':       "#ff00ff",
    'Re-capital':        "#005E05",
    'Re-inhabitation':   '#00bfff',
    'Re-transportation': '#ff0000'
}

epochs = list(range(1, len(train_acc_per_class) + 1))

def transpose_to_class_dict(data):
    return {cls: [epoch_data[i] for epoch_data in data] for i, cls in enumerate(classes_list)}

train_acc_class  = transpose_to_class_dict(train_acc_per_class)
val_acc_class    = transpose_to_class_dict(val_acc_per_class)

fig, axs = plt.subplots(3, 1, figsize=(10, 10))

for cls in classes_list:
    axs[0].plot(epochs, train_acc_class[cls], label=cls, color=color_map[cls])
axs[0].set_title('Training Accuracy per Category')
axs[0].set_xlabel('Epoch'); axs[0].set_ylabel('Accuracy (%)')
axs[0].set_xticks(epochs); axs[0].set_xlim(1, 30); axs[0].set_ylim(20, 100)

for cls in classes_list:
    axs[1].plot(epochs, val_acc_class[cls], label=cls, color=color_map[cls])
axs[1].set_title('Validation Accuracy per Category')
axs[1].set_xlabel('Epoch'); axs[1].set_ylabel('Accuracy (%)')
axs[1].set_xticks(epochs); axs[1].set_xlim(1, 30); axs[1].set_ylim(20, 100)

axs[2].plot(epochs, overall_train_acc, label='Training')
axs[2].plot(epochs, overall_val_acc,   label='Validation')
axs[2].set_title('Overall Accuracy')
axs[2].set_xlabel('Epoch'); axs[2].set_ylabel('Accuracy (%)')
axs[2].set_xticks(epochs); axs[2].set_xlim(1, 30); axs[2].set_ylim(20, 100)

for ax in axs.ravel():
    ax.grid(True)
    ax.legend(loc='lower right')
plt.tight_layout()
plt.show()